# 06. Content-Based шаг за шагом

Сначала вручную строим weighted sparse profile одного validation-пользователя. Только после этого используем технические функции для массового inference и других окон.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Mounted at /content/drive
Корень проекта: /content/drive/MyDrive/fashion-recommender-system


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем sparse operations, OneHotEncoder и cosine similarity  
**Зачем:** полная item-item matrix не нужна  
**Что получим:** библиотеки и технические inference-функции

In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display

from fashion_recommender.content_based import (
    ContentArtifacts, build_user_profiles, generate_content_candidates,
    seen_items_by_user,
)
from fashion_recommender.data import load_articles, load_transactions
from fashion_recommender.evaluation import (
    candidate_recall_at_k, hit_rate_at_k, map_at_k, mean_recall_at_k,
)
from fashion_recommender.persistence import load_json, save_content_artifacts, save_json

### Пути и параметры

**Что делаем:** задаём decay, candidate limit и reproducible cohort  
**Зачем:** те же параметры используются во всех окнах  
**Что получим:** пути и константы

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = PROCESSED_DIR / "temporal_windows.json"
if not WINDOWS_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден файл: {WINDOWS_PATH}\n"
        "Сначала выполните notebook 03_temporal_validation_colab.ipynb."
    )

DECAY_DAYS = 30.0
CANDIDATE_LIMIT = 50
MAX_EVALUATION_USERS = 2_000
RANDOM_STATE = 42

### Признаки товаров

**Что делаем:** явно задаём шесть объяснимых категорий  
**Зачем:** они определяют пространство content vectors  
**Что получим:** `ITEM_FEATURE_COLUMNS`

In [5]:
ITEM_FEATURE_COLUMNS = [
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "department_name",
    "section_name",
    "garment_group_name",
]
print(ITEM_FEATURE_COLUMNS)

['product_type_name', 'product_group_name', 'colour_group_name', 'department_name', 'section_name', 'garment_group_name']


### Загрузка входов

**Что делаем:** читаем transactions, articles и windows  
**Зачем:** item metadata статичны для всех temporal windows  
**Что получим:** три входных объекта

In [6]:
transactions = load_transactions(TRANSACTIONS_PATH)
articles = load_articles(ARTICLES_PATH)
windows = load_json(WINDOWS_PATH)
print("Transactions:", transactions.shape)
print("Articles:", articles.shape)

Transactions: (1048575, 5)
Articles: (105542, 25)


### Заполнение категорий

**Что делаем:** выбираем нужные столбцы и заменяем пропуски  
**Зачем:** OneHotEncoder должен получать строки без NaN  
**Что получим:** `article_features`

In [7]:
article_features = articles[
    ["article_id", *ITEM_FEATURE_COLUMNS]
].drop_duplicates("article_id").copy()
article_features[ITEM_FEATURE_COLUMNS] = (
    article_features[ITEM_FEATURE_COLUMNS]
    .fillna("Unknown")
    .astype(str)
)
display(article_features.head())

,article_id,product_type_name,product_group_name,colour_group_name,department_name,section_name,garment_group_name
0,0108775015,Vest top,Garment Upper body,Black,Jersey Basic,Womens Everyday Basics,Jersey Basic
1,0108775044,Vest top,Garment Upper body,White,Jersey Basic,Womens Everyday Basics,Jersey Basic
2,0108775051,Vest top,Garment Upper body,Off White,Jersey Basic,Womens Everyday Basics,Jersey Basic
3,0110065001,Bra,Underwear,Black,Clean Lingerie,Womens Lingerie,"Under-, Nightwear"
4,0110065002,Bra,Underwear,White,Clean Lingerie,Womens Lingerie,"Under-, Nightwear"


### Создание encoder

**Что делаем:** задаём OneHotEncoder до fit  
**Зачем:** неизвестная категория не должна ломать inference  
**Что получим:** `encoder`

In [8]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
)
print(encoder)

OneHotEncoder(handle_unknown='ignore')


### Sparse item matrix

**Что делаем:** вызываем `fit_transform` на категориях  
**Зачем:** матрица остаётся sparse  
**Что получим:** `article_matrix`

In [9]:
article_matrix = encoder.fit_transform(
    article_features[ITEM_FEATURE_COLUMNS]
)
article_matrix = article_matrix.tocsr().astype("float32")
print("Type:", type(article_matrix))

Type: <class 'scipy.sparse._csr.csr_matrix'>


### Размер item matrix

**Что делаем:** смотрим shape, nnz и имена первых one-hot признаков  
**Зачем:** число строк должно совпадать с числом товаров  
**Что получим:** проверку encoder output

In [10]:
encoded_feature_names = encoder.get_feature_names_out(ITEM_FEATURE_COLUMNS)
print("Shape:", article_matrix.shape)
print("NNZ:", article_matrix.nnz)
print("Первые признаки:", encoded_feature_names[:10])
assert article_matrix.shape[0] == len(article_features)

Shape: (105542, 527)
NNZ: 633252
Первые признаки: ['product_type_name_Accessories set' 'product_type_name_Alice band'
 'product_type_name_Baby Bib' 'product_type_name_Backpack'
 'product_type_name_Bag' 'product_type_name_Ballerinas'
 'product_type_name_Beanie' 'product_type_name_Belt'
 'product_type_name_Bikini top' 'product_type_name_Blanket']


### Article mapping

**Что делаем:** связываем article ID со строкой matrix  
**Зачем:** после cosine numeric index нужно вернуть в article ID  
**Что получим:** `article_to_index` и `index_to_article`

In [11]:
index_to_article = article_features["article_id"].astype(str).tolist()
article_to_index = {
    article_id: index
    for index, article_id in enumerate(index_to_article)
}
print("Mapping example:", next(iter(article_to_index.items())))

Mapping example: ('0108775015', 0)


### Content artifacts в памяти

**Что делаем:** упаковываем encoder, matrix и mapping  
**Зачем:** технические функции массового inference принимают один согласованный объект  
**Что получим:** `content_artifacts`

In [12]:
content_artifacts = ContentArtifacts(
    encoder=encoder,
    article_feature_matrix=article_matrix,
    article_to_index=article_to_index,
    index_to_article=index_to_article,
    feature_columns=ITEM_FEATURE_COLUMNS,
)

### Validation-границы

**Что делаем:** выбираем одно окно для подробного профиля  
**Зачем:** weights должны измеряться относительно validation cutoff  
**Что получим:** две даты

In [13]:
validation_cutoff = pd.Timestamp(windows["validation"]["cutoff_date"])
validation_end = pd.Timestamp(windows["validation"]["target_end_date"])
print("Validation:", validation_cutoff.date(), "—", validation_end.date())

Validation: 2019-12-18 — 2019-12-24


### Validation history и target

**Что делаем:** делим транзакции по cutoff  
**Зачем:** target не участвует в профилях  
**Что получим:** две таблицы

In [14]:
validation_history = transactions[
    transactions["t_dat"] < validation_cutoff
].copy()
validation_target = transactions[
    transactions["t_dat"].between(validation_cutoff, validation_end)
].copy()

assert validation_history["t_dat"].max() < validation_cutoff
print("History:", validation_history.shape, "Target:", validation_target.shape)

History: (1011880, 5) Target: (23405, 5)


### Validation ground truth

**Что делаем:** оставляем известных users и собираем уникальные future items  
**Зачем:** content и ALS должны сравниваться на одном protocol  
**Что получим:** полный `validation_ground_truth`

In [15]:
validation_known_users = set(validation_history["customer_id"])
validation_target_evaluation = validation_target[
    validation_target["customer_id"].isin(validation_known_users)
]
validation_target_unique = (
    validation_target_evaluation
    .sort_values("t_dat")
    .drop_duplicates(["customer_id", "article_id"])
)
validation_ground_truth = validation_target_unique.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
print("Ground-truth users:", len(validation_ground_truth))

Ground-truth users: 13078


### Validation cohort

**Что делаем:** выбираем ID случайно с тем же seed  
**Зачем:** candidate files ALS и Content должны иметь один cohort  
**Что получим:** sample ground truth

In [16]:
validation_all_users = np.array(sorted(validation_ground_truth))
validation_sample_size = min(MAX_EVALUATION_USERS, len(validation_all_users))
validation_rng = np.random.default_rng(RANDOM_STATE)
validation_users = validation_rng.choice(
    validation_all_users,
    size=validation_sample_size,
    replace=False,
).tolist()
validation_ground_truth_sample = {
    customer_id: validation_ground_truth[customer_id]
    for customer_id in validation_users
}
print("Evaluation users:", len(validation_users))

Evaluation users: 2000


### Purchase count

**Что делаем:** считаем частоту каждой user-item пары  
**Зачем:** frequency weight использует число повторов  
**Что получим:** `validation_purchase_counts`

In [17]:
validation_purchase_counts = (
    validation_history
    .groupby(["customer_id", "article_id"])
    .size()
    .reset_index(name="purchase_count")
)
display(validation_purchase_counts.head())

,customer_id,article_id,purchase_count
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,1
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,1
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,1
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,1
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,1


### Последняя покупка пары

**Что делаем:** отдельно находим максимальную дату user-item  
**Зачем:** recency зависит от последнего события  
**Что получим:** `validation_last_purchases`

In [18]:
validation_last_purchases = (
    validation_history
    .groupby(["customer_id", "article_id"], as_index=False)["t_dat"]
    .max()
    .rename(columns={"t_dat": "last_purchase"})
)
display(validation_last_purchases.head())

,customer_id,article_id,last_purchase
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,2019-05-25
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,2019-09-28
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,2019-08-12
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,2019-05-22
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,2019-06-18


### User-item history

**Что делаем:** объединяем count и last date  
**Зачем:** два сигнала остаются видимыми отдельными столбцами  
**Что получим:** `validation_user_item_history`

In [19]:
validation_user_item_history = validation_purchase_counts.merge(
    validation_last_purchases,
    on=["customer_id", "article_id"],
    how="inner",
)
validation_user_item_history = validation_user_item_history[
    validation_user_item_history["article_id"].isin(article_to_index)
].copy()
display(validation_user_item_history.head())

,customer_id,article_id,purchase_count,last_purchase
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,1,2019-05-25
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,1,2019-09-28
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,1,2019-08-12
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,1,2019-05-22
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,1,2019-06-18


### Days since purchase

**Что делаем:** вычитаем last purchase из cutoff  
**Зачем:** future dates здесь недопустимы  
**Что получим:** неотрицательный `days_since_purchase`

In [20]:
validation_user_item_history["days_since_purchase"] = (
    validation_cutoff
    - validation_user_item_history["last_purchase"]
).dt.days
assert validation_user_item_history["days_since_purchase"].min() >= 0
display(validation_user_item_history.head())

,customer_id,article_id,purchase_count,last_purchase,days_since_purchase
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601006,1,2019-05-25,207
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0797065001,1,2019-09-28,81
2,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0611584007,1,2019-08-12,128
3,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0640021005,1,2019-05-22,210
4,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0689898002,1,2019-06-18,183


### Recency weight

**Что делаем:** экспоненциально уменьшаем вклад старой покупки  
**Зачем:** недавние товары сильнее влияют на профиль  
**Что получим:** столбец `recency_weight`

In [21]:
validation_user_item_history["recency_weight"] = np.exp(
    -validation_user_item_history["days_since_purchase"] / DECAY_DAYS
)
display(validation_user_item_history[
    ["days_since_purchase", "recency_weight"]
].head())

,days_since_purchase,recency_weight
0,207,0.001008
1,81,0.067206
2,128,0.014028
3,210,0.000912
4,183,0.002243


### Frequency weight

**Что делаем:** логарифмически усиливаем повторные покупки  
**Зачем:** один очень частый товар не должен полностью доминировать  
**Что получим:** столбец `frequency_weight`

In [22]:
validation_user_item_history["frequency_weight"] = (
    1 + np.log1p(validation_user_item_history["purchase_count"])
)
display(validation_user_item_history[
    ["purchase_count", "frequency_weight"]
].head())

,purchase_count,frequency_weight
0,1,1.693147
1,1,1.693147
2,1,1.693147
3,1,1.693147
4,1,1.693147


### Итоговый вес

**Что делаем:** перемножаем recency и frequency  
**Зачем:** профиль учитывает оба объяснимых сигнала  
**Что получим:** `purchase_weight`

In [23]:
validation_user_item_history["purchase_weight"] = (
    validation_user_item_history["recency_weight"]
    * validation_user_item_history["frequency_weight"]
)
display(validation_user_item_history[
    ["purchase_count", "days_since_purchase", "purchase_weight"]
].sample(5, random_state=RANDOM_STATE))

,purchase_count,days_since_purchase,purchase_weight
676781,1,80,0.117646
265570,1,65,0.193965
249502,1,96,0.069016
911155,1,240,0.000568
491472,1,259,0.000301


### Один подходящий пользователь

**Что делаем:** выбираем evaluation user с известными article rows  
**Зачем:** профиль сначала разбирается на одном примере  
**Что получим:** `example_customer`

In [24]:
eligible_profile_users = set(validation_user_item_history["customer_id"])
example_customer = None
for customer_id in validation_users:
    if customer_id in eligible_profile_users:
        example_customer = customer_id
        break

if example_customer is None:
    raise ValueError("Не найден пользователь для примера профиля")
print("Example customer:", example_customer)

Example customer: f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773caf840ca3abcd7fc9de


### Покупки пользователя

**Что делаем:** фильтруем weighted user-item history  
**Зачем:** видим товары, даты, counts и weights  
**Что получим:** `example_purchases`

In [25]:
example_purchases = validation_user_item_history[
    validation_user_item_history["customer_id"] == example_customer
].copy()
example_purchases["article_index"] = example_purchases["article_id"].map(
    article_to_index
)
display(example_purchases)

,customer_id,article_id,purchase_count,last_purchase,days_since_purchase,recency_weight,frequency_weight,purchase_weight,article_index
951523,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0464297033,1,2019-07-19,152,0.006303,1.693147,0.010673,3718
951524,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0470789001,1,2019-09-08,101,0.034504,1.693147,0.058421,3940
951525,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0633130008,1,2019-01-14,338,0.000013,1.693147,0.000022,31683
951526,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0697813039,1,2019-07-19,152,0.006303,1.693147,0.010673,50935


### Строки item matrix

**Что делаем:** выбираем sparse vectors купленных товаров  
**Зачем:** матрица не превращается в dense  
**Что получим:** `example_item_rows`

In [26]:
example_indices = example_purchases["article_index"].to_numpy()
example_item_rows = article_matrix[example_indices]
print("Item rows shape:", example_item_rows.shape)
print("Item rows nnz:", example_item_rows.nnz)

Item rows shape: (4, 527)
Item rows nnz: 24


### Веса примера

**Что делаем:** берём purchase weights в том же порядке  
**Зачем:** каждый item vector получает свой коэффициент  
**Что получим:** sparse row с весами

In [27]:
example_weights = example_purchases["purchase_weight"].to_numpy(
    dtype="float32"
)
example_weight_row = csr_matrix(example_weights.reshape(1, -1))
print("Weights:", example_weights)
print("Weight sum:", example_weights.sum())

Weights: [1.0672578e-02 5.8421131e-02 2.1659256e-05 1.0672578e-02]
Weight sum: 0.07978795


### Взвешенная сумма

**Что делаем:** умножаем weights на item rows  
**Зачем:** получаем сумму признаков без dense conversion  
**Что получим:** `example_weighted_sum`

In [28]:
example_weighted_sum = example_weight_row @ example_item_rows
print("Weighted sum shape:", example_weighted_sum.shape)
print("Weighted sum nnz:", example_weighted_sum.nnz)

Weighted sum shape: (1, 527)
Weighted sum nnz: 16


### User profile

**Что делаем:** делим сумму признаков на сумму весов  
**Зачем:** профиль становится взвешенным средним  
**Что получим:** sparse `example_profile`

In [29]:
example_profile = example_weighted_sum / example_weights.sum()
example_profile = example_profile.tocsr()
print("Profile shape:", example_profile.shape)
print("Profile nnz:", example_profile.nnz)

Profile shape: (1, 527)
Profile nnz: 16


### Cosine similarity

**Что делаем:** сравниваем один profile со всеми item rows  
**Зачем:** не строим квадратную item-item matrix  
**Что получим:** один similarity vector

In [30]:
example_similarities = cosine_similarity(
    example_profile,
    article_matrix,
).ravel()
print("Similarities shape:", example_similarities.shape)
print("Max similarity:", example_similarities.max())

Similarities shape: (105542,)
Max similarity: 0.983968266094377


### Исключение seen items

**Что делаем:** понижаем score уже купленных товаров  
**Зачем:** кандидаты должны предлагать новые позиции  
**Что получим:** filtered similarity vector

In [31]:
example_seen_items = set(example_purchases["article_id"])
example_seen_indices = [
    article_to_index[article_id]
    for article_id in example_seen_items
]
example_similarities[example_seen_indices] = -np.inf
print("Исключено товаров:", len(example_seen_indices))

Исключено товаров: 4


### Top-10 похожих товаров

**Что делаем:** сортируем similarity и возвращаем article ID  
**Зачем:** завершаем ручной путь одного пользователя  
**Что получим:** таблицу Top-10

In [32]:
example_top_indices = np.argsort(example_similarities)[-10:][::-1]
example_top_articles = [
    index_to_article[int(item_index)]
    for item_index in example_top_indices
]
example_top10 = pd.DataFrame({
    "article_id": example_top_articles,
    "cosine_similarity": example_similarities[example_top_indices],
})
display(example_top10)

,article_id,cosine_similarity
0,0834906002,0.983968
1,0470789028,0.983968
2,0704754001,0.983968
3,0901965001,0.983968
4,0897682001,0.983968
5,0866657001,0.983968
6,0863477001,0.983968
7,0863515001,0.983968
8,0507883009,0.983968
9,0792023001,0.983968


### Все validation profiles

**Что делаем:** повторяем показанные weights и sparse aggregation для всех users  
**Зачем:** функция не выполняет Top-K  
**Что получим:** `validation_profiles`

In [33]:
validation_profiles = build_user_profiles(
    validation_history,
    content_artifacts,
    reference_date=validation_cutoff,
    decay_days=DECAY_DAYS,
)
print("Profiles shape:", validation_profiles.matrix.shape)

Profiles shape: (447850, 527)


### Validation seen items

**Что делаем:** собираем историю покупок каждого user  
**Зачем:** mass inference исключит эти article ID  
**Что получим:** `validation_seen_items`

In [34]:
validation_seen_items = seen_items_by_user(validation_history)
print("Users with seen items:", len(validation_seen_items))

Users with seen items: 447850


### Mass validation candidates

**Что делаем:** вызываем batched user-to-item cosine  
**Зачем:** ручной алгоритм одного user уже показан  
**Что получим:** `validation_candidates`

In [35]:
validation_candidates = generate_content_candidates(
    validation_profiles,
    content_artifacts,
    customer_ids=validation_users,
    seen_items=validation_seen_items,
    limit=CANDIDATE_LIMIT,
)
print("Candidate rows:", len(validation_candidates))
display(validation_candidates.head())

Candidate rows: 100000


,customer_id,article_id,content_similarity_score,content_rank
0,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0470789028,0.983968,1
1,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0471077001,0.983968,2
2,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0473954006,0.983968,3
3,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0507883009,0.983968,4
4,f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...,0555767002,0.983968,5


### Validation Candidate Recall

**Что делаем:** оцениваем покрытие Top-50  
**Зачем:** это верхняя граница для последующего ranking  
**Что получим:** Candidate Recall

In [36]:
validation_candidate_lists = validation_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
validation_candidate_recall = candidate_recall_at_k(
    validation_ground_truth_sample,
    validation_candidate_lists,
    CANDIDATE_LIMIT,
)
print(f"Candidate Recall@{CANDIDATE_LIMIT}:", validation_candidate_recall)

Candidate Recall@50: 0.00675


### Validation Top-12

**Что делаем:** оцениваем standalone Content-Based  
**Зачем:** позиционные метрики рассчитаны отдельно от сохранения  
**Что получим:** Recall/MAP/HitRate

In [37]:
validation_top12_lists = validation_candidates[
    validation_candidates["content_rank"] <= 12
].groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()

validation_metrics = {
    "Recall@12": mean_recall_at_k(validation_ground_truth_sample, validation_top12_lists, 12),
    "MAP@12": map_at_k(validation_ground_truth_sample, validation_top12_lists, 12),
    "HitRate@12": hit_rate_at_k(validation_ground_truth_sample, validation_top12_lists, 12),
}
display(pd.Series(validation_metrics))

,0
Recall@12,0.002000
MAP@12,0.000322
HitRate@12,0.002000


### Сохранение validation candidates

**Что делаем:** записываем отдельный Parquet  
**Зачем:** notebook 07 загрузит готовый результат  
**Что получим:** `content_candidates_validation.parquet`

In [38]:
validation_candidates_path = PROCESSED_DIR / "content_candidates_validation.parquet"
validation_candidates.to_parquet(validation_candidates_path, index=False)
print("Сохранено:", validation_candidates_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/content_candidates_validation.parquet


### Ground truth helper

**Что делаем:** фиксируем уже разобранную pandas-логику  
**Зачем:** функция не строит profile и не считает cosine  
**Что получим:** `build_ground_truth`

In [39]:
def build_ground_truth(target, known_users):
    target_evaluation = target[target["customer_id"].isin(known_users)]
    target_unique = (
        target_evaluation
        .sort_values("t_dat")
        .drop_duplicates(["customer_id", "article_id"])
    )
    return target_unique.groupby(
        "customer_id", sort=False
    )["article_id"].apply(list).to_dict()

### Sampling helper

**Что делаем:** повторяем deterministic cohort  
**Зачем:** полный ground truth остаётся отдельным объектом  
**Что получим:** users и sample dictionary

In [40]:
def sample_ground_truth(ground_truth, max_users, random_state):
    all_users = np.array(sorted(ground_truth))
    sample_size = min(max_users, len(all_users))
    rng = np.random.default_rng(random_state)
    users = rng.choice(all_users, size=sample_size, replace=False).tolist()
    ground_truth_sample = {
        customer_id: ground_truth[customer_id]
        for customer_id in users
    }
    return users, ground_truth_sample

### Train: границы

**Что делаем:** выбираем train-окно  
**Зачем:** validation уже показал расчёт весов и одного профиля  
**Что получим:** `train_cutoff` и `train_end`

In [41]:
train_cutoff = pd.Timestamp(windows["train"]["cutoff_date"])
train_end = pd.Timestamp(windows["train"]["target_end_date"])
print("Train:", train_cutoff.date(), "—", train_end.date())

Train: 2019-12-11 — 2019-12-17


### Train: history и target

**Что делаем:** разделяем прошлое и будущую неделю  
**Зачем:** content profiles используют только history  
**Что получим:** `train_history` и `train_target`

In [42]:
train_history = transactions[
    transactions["t_dat"] < train_cutoff
].copy()
train_target = transactions[
    transactions["t_dat"].between(train_cutoff, train_end)
].copy()

assert train_history["t_dat"].max() < train_cutoff
print("History:", train_history.shape, "Target:", train_target.shape)

History: (998087, 5) Target: (13793, 5)


### Train: ground truth

**Что делаем:** повторяем уже показанную подготовку ответов  
**Зачем:** оценка использует полный target каждого пользователя  
**Что получим:** `train_ground_truth`

In [43]:
train_ground_truth = build_ground_truth(
    train_target,
    set(train_history["customer_id"]),
)
print("Ground-truth users:", len(train_ground_truth))

Ground-truth users: 7812


### Train: cohort

**Что делаем:** выбираем пользователей с тем же seed, что в ALS  
**Зачем:** candidate sources должны покрывать одинаковый cohort  
**Что получим:** `train_users` и sample ground truth

In [44]:
train_users, train_ground_truth_sample = sample_ground_truth(
    train_ground_truth,
    MAX_EVALUATION_USERS,
    RANDOM_STATE,
)
print("Evaluation users:", len(train_users))

Evaluation users: 2000


### Train: profile matrix

**Что делаем:** повторяем показанные weights и sparse aggregation  
**Зачем:** техническая функция не выполняет cosine или Top-K  
**Что получим:** `train_profiles`

In [45]:
train_profiles = build_user_profiles(
    train_history,
    content_artifacts,
    reference_date=train_cutoff,
    decay_days=DECAY_DAYS,
)
print("Profiles:", train_profiles.matrix.shape)

Profiles: (443629, 527)


### Train: seen items

**Что делаем:** собираем покупки, которые нельзя рекомендовать повторно  
**Зачем:** filtering отделён от profile building  
**Что получим:** `train_seen_items`

In [46]:
train_seen_items = seen_items_by_user(train_history)
print("Users with seen items:", len(train_seen_items))

Users with seen items: 443629


### Train: массовые кандидаты

**Что делаем:** считаем user-to-items cosine batched-функцией  
**Зачем:** один пользователь уже был полностью разобран вручную  
**Что получим:** `train_candidates`

In [47]:
train_candidates = generate_content_candidates(
    train_profiles,
    content_artifacts,
    customer_ids=train_users,
    seen_items=train_seen_items,
    limit=CANDIDATE_LIMIT,
)
print("Candidate rows:", len(train_candidates))
display(train_candidates.head())

Candidate rows: 100000


,customer_id,article_id,content_similarity_score,content_rank
0,565e138a74f621194fba850fd52c43c2fd462714437748...,0497247004,0.672343,1
1,565e138a74f621194fba850fd52c43c2fd462714437748...,0506205001,0.672343,2
2,565e138a74f621194fba850fd52c43c2fd462714437748...,0508457002,0.672343,3
3,565e138a74f621194fba850fd52c43c2fd462714437748...,0523391002,0.672343,4
4,565e138a74f621194fba850fd52c43c2fd462714437748...,0523465002,0.672343,5


### Train: метрики

**Что делаем:** считаем Candidate Recall и standalone Top-12  
**Зачем:** метрики не смешаны с inference или сохранением  
**Что получим:** `train_metrics`

In [48]:
train_candidate_lists = train_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
train_top12_lists = train_candidates[
    train_candidates["content_rank"] <= 12
].groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()
train_metrics = {
    "Candidate Recall": candidate_recall_at_k(
        train_ground_truth_sample, train_candidate_lists, CANDIDATE_LIMIT
    ),
    "Recall@12": mean_recall_at_k(train_ground_truth_sample, train_top12_lists, 12),
    "MAP@12": map_at_k(train_ground_truth_sample, train_top12_lists, 12),
    "HitRate@12": hit_rate_at_k(train_ground_truth_sample, train_top12_lists, 12),
}
display(pd.Series(train_metrics))

,0
Candidate Recall,0.003000
Recall@12,0.000250
MAP@12,0.000021
HitRate@12,0.000500


### Train: сохранение кандидатов

**Что делаем:** записываем готовую candidate table  
**Зачем:** notebook 07 не будет строить profiles повторно  
**Что получим:** `content_candidates_train.parquet`

In [49]:
train_candidates_path = PROCESSED_DIR / "content_candidates_train.parquet"
train_candidates.to_parquet(train_candidates_path, index=False)
print("Сохранено:", train_candidates_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/content_candidates_train.parquet


### Test: границы

**Что делаем:** выбираем test-окно  
**Зачем:** validation уже показал расчёт весов и одного профиля  
**Что получим:** `test_cutoff` и `test_end`

In [50]:
test_cutoff = pd.Timestamp(windows["test"]["cutoff_date"])
test_end = pd.Timestamp(windows["test"]["target_end_date"])
print("Test:", test_cutoff.date(), "—", test_end.date())

Test: 2019-12-25 — 2019-12-31


### Test: history и target

**Что делаем:** разделяем прошлое и будущую неделю  
**Зачем:** content profiles используют только history  
**Что получим:** `test_history` и `test_target`

In [51]:
test_history = transactions[
    transactions["t_dat"] < test_cutoff
].copy()
test_target = transactions[
    transactions["t_dat"].between(test_cutoff, test_end)
].copy()

assert test_history["t_dat"].max() < test_cutoff
print("History:", test_history.shape, "Target:", test_target.shape)

History: (1035285, 5) Target: (13290, 5)


### Test: ground truth

**Что делаем:** повторяем уже показанную подготовку ответов  
**Зачем:** оценка использует полный target каждого пользователя  
**Что получим:** `test_ground_truth`

In [52]:
test_ground_truth = build_ground_truth(
    test_target,
    set(test_history["customer_id"]),
)
print("Ground-truth users:", len(test_ground_truth))

Ground-truth users: 7590


### Test: cohort

**Что делаем:** выбираем пользователей с тем же seed, что в ALS  
**Зачем:** candidate sources должны покрывать одинаковый cohort  
**Что получим:** `test_users` и sample ground truth

In [53]:
test_users, test_ground_truth_sample = sample_ground_truth(
    test_ground_truth,
    MAX_EVALUATION_USERS,
    RANDOM_STATE,
)
print("Evaluation users:", len(test_users))

Evaluation users: 2000


### Test: profile matrix

**Что делаем:** повторяем показанные weights и sparse aggregation  
**Зачем:** техническая функция не выполняет cosine или Top-K  
**Что получим:** `test_profiles`

In [54]:
test_profiles = build_user_profiles(
    test_history,
    content_artifacts,
    reference_date=test_cutoff,
    decay_days=DECAY_DAYS,
)
print("Profiles:", test_profiles.matrix.shape)

Profiles: (454441, 527)


### Test: seen items

**Что делаем:** собираем покупки, которые нельзя рекомендовать повторно  
**Зачем:** filtering отделён от profile building  
**Что получим:** `test_seen_items`

In [55]:
test_seen_items = seen_items_by_user(test_history)
print("Users with seen items:", len(test_seen_items))

Users with seen items: 454441


### Test: массовые кандидаты

**Что делаем:** считаем user-to-items cosine batched-функцией  
**Зачем:** один пользователь уже был полностью разобран вручную  
**Что получим:** `test_candidates`

In [56]:
test_candidates = generate_content_candidates(
    test_profiles,
    content_artifacts,
    customer_ids=test_users,
    seen_items=test_seen_items,
    limit=CANDIDATE_LIMIT,
)
print("Candidate rows:", len(test_candidates))
display(test_candidates.head())

Candidate rows: 100000


,customer_id,article_id,content_similarity_score,content_rank
0,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0416511016,0.717103,1
1,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0416511021,0.717103,2
2,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0448509018,0.717103,3
3,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0448509028,0.717103,4
4,5883d17360e658d033e6c3e8b96a06b9cf5cda86bc1b2b...,0448515019,0.717103,5


### Test: метрики

**Что делаем:** считаем Candidate Recall и standalone Top-12  
**Зачем:** метрики не смешаны с inference или сохранением  
**Что получим:** `test_metrics`

In [57]:
test_candidate_lists = test_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
test_top12_lists = test_candidates[
    test_candidates["content_rank"] <= 12
].groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()
test_metrics = {
    "Candidate Recall": candidate_recall_at_k(
        test_ground_truth_sample, test_candidate_lists, CANDIDATE_LIMIT
    ),
    "Recall@12": mean_recall_at_k(test_ground_truth_sample, test_top12_lists, 12),
    "MAP@12": map_at_k(test_ground_truth_sample, test_top12_lists, 12),
    "HitRate@12": hit_rate_at_k(test_ground_truth_sample, test_top12_lists, 12),
}
display(pd.Series(test_metrics))

,0
Candidate Recall,0.003500
Recall@12,0.001000
MAP@12,0.000292
HitRate@12,0.001000


### Test: сохранение кандидатов

**Что делаем:** записываем готовую candidate table  
**Зачем:** notebook 07 не будет строить profiles повторно  
**Что получим:** `content_candidates_test.parquet`

In [58]:
test_candidates_path = PROCESSED_DIR / "content_candidates_test.parquet"
test_candidates.to_parquet(test_candidates_path, index=False)
print("Сохранено:", test_candidates_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/content_candidates_test.parquet


### Test alias

**Что делаем:** сохраняем совместимое имя test candidates  
**Зачем:** старые consumers не требуют изменения пути  
**Что получим:** `content_candidates.parquet`

In [59]:
content_alias_path = PROCESSED_DIR / "content_candidates.parquet"
test_candidates.to_parquet(content_alias_path, index=False)
print("Сохранено:", content_alias_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/content_candidates.parquet


### Сохранение encoder и matrix

**Что делаем:** записываем статичные content artifacts  
**Зачем:** batch inference не должен повторно fit OneHotEncoder  
**Что получим:** joblib, NPZ, mapping и config

In [60]:
content_paths = save_content_artifacts(
    encoder,
    article_matrix,
    index_to_article,
    MODEL_DIR,
    {"feature_columns": ITEM_FEATURE_COLUMNS, "decay_days": DECAY_DAYS},
)
print(content_paths)

{'encoder': PosixPath('/content/drive/MyDrive/fashion-recommender-system/models/content_encoder.joblib'), 'matrix': PosixPath('/content/drive/MyDrive/fashion-recommender-system/models/article_feature_matrix.npz'), 'article_ids': PosixPath('/content/drive/MyDrive/fashion-recommender-system/models/mappings/content_article_ids.json'), 'config': PosixPath('/content/drive/MyDrive/fashion-recommender-system/models/content_config.json')}


### Сохранение явного mapping

**Что делаем:** записываем article ID → row index  
**Зачем:** mapping можно проверить без загрузки matrix  
**Что получим:** `article_index_mapping.json`

In [61]:
article_mapping_path = MODEL_DIR / "mappings" / "article_index_mapping.json"
save_json(article_to_index, article_mapping_path)
print("Сохранено:", article_mapping_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/models/mappings/article_index_mapping.json


### Content test report

**Что делаем:** добавляем model name и статистику к test metrics  
**Зачем:** notebook 10 загрузит компактный CSV  
**Что получим:** `content_metrics.csv`

In [62]:
content_test_report = {
    "model": "Content-Based",
    **test_metrics,
    "users_evaluated": len(test_ground_truth_sample),
    "average_candidates": test_candidates.groupby("customer_id").size().mean(),
    "notes": f"Content-Based Top-{CANDIDATE_LIMIT} candidates",
}
content_metrics_path = REPORT_DIR / "content_metrics.csv"
pd.DataFrame([content_test_report]).to_csv(content_metrics_path, index=False)
display(pd.Series(content_test_report))

,0
model,Content-Based
Candidate Recall,0.0035
Recall@12,0.001
MAP@12,0.000292
HitRate@12,0.001
users_evaluated,2000
average_candidates,50.0
notes,Content-Based Top-50 candidates
